First connect to the Docent server and web UI.

In [ ]:
from docent import DocentClient
client = DocentClient(server_url="http://localhost:8889", web_url="http://localhost:3001")

You can think of each `FrameGrid` as a collection of transcripts. We'll create a fresh one.

In [ ]:
fg_id = client.create_framegrid(name="sample framegrid", description="example that comes with the Docent repo")

Now we're ready to ingest some logs. We've provided a `sample_transcript.json` file in this directory. It's a log from OpenHands' run on SWE-Bench.

First, we write a function that takes the JSON and parses it into a `Transcript` object, complete with `TranscriptMetadata`. You can click into the definitions of these objects to understand their schemas.

In [ ]:
import json
from docent._frames.transcript import Transcript, TranscriptMetadata
from docent._llm_util.types import (
    ChatMessage,
    ChatMessageAssistant,
    ChatMessageTool,
    ChatMessageUser,
    ToolCall,
)

def load_openhands_swebench_log(file: str) -> Transcript:
    with open(file, "r") as f:
        data = json.load(f)

    messages: list[ChatMessage] = []

    for msg in data["history"]:
        source = msg["source"]
        action, observation = msg.get("action"), msg.get("observation")
        tc_metadata = msg.get("tool_call_metadata")

        if action and observation:
            raise ValueError("Action and observation cannot both be present")

        if source == "user":
            cur_msg = ChatMessageUser(content=msg["message"])
        elif source == "agent":
            if not tc_metadata:
                print(
                    f"Tool call metadata is required for agent messages. Message:\n{json.dumps(msg, indent=2)}"
                )
                continue
            if action:  # Assistant with tool call
                response = tc_metadata["model_response"]["choices"][0]["message"]
                tool_calls: list[ToolCall] = []
                for tc in response["tool_calls"]:
                    try:
                        tc_args = json.loads(tc["function"]["arguments"])
                        parse_error = None
                    except Exception as e:
                        tc_args = {"arguments": tc["function"]["arguments"]}
                        parse_error = str(e)

                    tool_calls.append(
                        ToolCall(
                            id=tc["id"],
                            function=tc["function"]["name"],
                            arguments=tc_args,
                            type="function",
                            parse_error=parse_error,
                        )
                    )
                cur_msg = ChatMessageAssistant(
                    content=response["content"] or "", tool_calls=tool_calls
                )
            elif observation:  # Tool response
                cur_msg = ChatMessageTool(
                    tool_call_id=tc_metadata["tool_call_id"],
                    function=tc_metadata["function_name"],
                    content=msg["content"],
                )
            else:
                raise ValueError("Neither observation nor action present")

        else:
            raise ValueError(f"Unknown source: {source}")

        messages.append(cur_msg)

    # Build metadata
    metadata = TranscriptMetadata(
        task_id="swe_bench_verified",
        sample_id=data["instance_id"],
        original_sample_id_type="str" if isinstance(data["instance_id"], str) else "int",
        epoch_id=1,  # Default to epoch 1
        experiment_id="default",
        intervention_description=None,
        intervention_timestamp=None,
        intervention_index=None,
        model=data["metadata"]["llm_config"]["model"],
        task_args={},
        is_loading_messages=False,
        scores={"resolved": data["report"]["resolved"]},
        default_score_key="resolved",
        additional_metadata={},
        scoring_metadata=data["instance"],
    )

    # Create the transcript
    transcript = Transcript(
        messages=messages,
        metadata=metadata,
    )

    return transcript

Let's just load the single transcript in, and print its string representation.

In [ ]:
transcripts = [
    load_openhands_swebench_log(f) for f in ["sample_transcript.json"]
]
print(transcripts[0].to_str())

We can finally ingest the transcript and watch the UI update. If you navigate to the frontend URL printed above, you should see the transcript available for viewing.

In [ ]:
from docent._frames.types import Datapoint

client.add_datapoints(fg_id, [Datapoint.from_transcript(t) for t in transcripts])